# Stage 2 — Regularized MLP Models, Optuna Tuning, Ablations & Calibrated Uncertainty
**Deep Learning Capstone — Phase 2 (Member 2: MLP Architecture, Training & Uncertainty)**  
**Target:** Continuous Canadian Council of Ministers of the Environment Water Quality Index (`CCME_Values` $\in [31.3041, 100.0]$)  
**Objective:** Build a compact, heavily regularized Multi-Layer Perceptron (MLP) challenger on the exact leak-safe site-held-out split from Phase 1, verify model integrity via a 4-part Sanity Suite, match the AML 15-trial Optuna tuning budget on validation RMSE only, conduct a 20-configuration architectural ablation study, and close the **calibrated uncertainty gap** via Monte Carlo Dropout ($T=50$), a 5-Seed Deep Ensemble, and a Pinball Loss Quantile Regression network ($q_{05}, q_{50}, q_{95}$).

In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

dl_dir = None
for base in [Path.cwd(), *Path.cwd().parents]:
    if (base / "dl" / "src" / "models.py").exists():
        dl_dir = base / "dl"
        break
    if (base / "src" / "models.py").exists() and base.name == "dl":
        dl_dir = base
        break

if dl_dir and str(dl_dir) not in sys.path:
    sys.path.insert(0, str(dl_dir))

DL_ROOT = dl_dir
from src.utils import load_config, seed_everything
from src.models import WaterQualityMLP, count_parameters, build_mlp_from_config

seed_everything(42)
cfg = load_config()
model = build_mlp_from_config(cfg, input_dim=len(cfg["data"]["feature_list"]))
print(f"Loaded Phase 2 MLP config | Architecture: {model.hidden_dims} | Trainable Parameters: {count_parameters(model):,}")

## 1. Pre-Tuning Sanity Suite (Task C)
Before running hyperparameter search, we verify four foundational checks:
1. **32-Sample Mini-Batch Overfit:** Confirms gradients flow cleanly and drive training MSE to $\approx 0.0$ ($R^2 = 1.0$).
2. **Shuffled-Target Permutation Check:** Confirms test $R^2 \approx 0.0$ when `y_train` is randomly permuted (proving zero feature-target leakage).
3. **Trivial Baselines (Mean & Ridge):** Establishes the linear reference floor on the exact scaled splits.
4. **Initial Baseline MLP & Learning Curves:** Tracks Huber loss and validation RMSE across 50 epochs.

In [2]:
sanity_path = DL_ROOT / "results" / "tables" / "sanity_suite_summary.json"
with open(sanity_path, "r", encoding="utf-8") as f:
    sanity = json.load(f)

print("1. 32-Sample Overfit Check:", sanity["tiny_batch_overfit"])
print("2. Shuffled-Target Leakage Check:", sanity["shuffled_target_test"])
print("3. Trivial Baselines (Test):")
print("   - Train-Mean Predictor:", sanity["trivial_baselines"]["mean_predictor"]["test"])
print("   - Ridge Regression (21 features):", sanity["trivial_baselines"]["ridge_regression"]["test"])
print("4. Initial Baseline MLP (13,441 params):")
print("   - Validation:", sanity["initial_mlp_run"]["val_metrics"])
print("   - Test:      ", sanity["initial_mlp_run"]["test_metrics"])

curves_img = DL_ROOT / "results" / "figures" / "mlp_baseline_learning_curves.png"
if curves_img.exists():
    display(Image(filename=str(curves_img)))

## 2. Optuna Hyperparameter Search (Matched to AML 15-Trial Budget)
We matched the exact tuning budget of the AML baseline (**15 Optuna trials**) optimizing **Validation RMSE only** (`X_test` is never touched during tuning).

In [3]:
df_trials = pd.read_csv(DL_ROOT / "results" / "tables" / "optuna_trials.csv")
display(df_trials[["trial", "hidden_dims", "activation", "batch_norm", "dropout", "learning_rate", "weight_decay", "batch_size", "n_parameters", "val_RMSE", "val_MAE", "val_R2"]].head(10))

## 3. Systematic Architecture & Regularization Ablation Table (Task D)
We systematically ablate **Depth** (1–4 layers), **Width** (32–256 units), **Dropout** ($0.0, 0.1, 0.3$), **BatchNorm** (On vs. Off), **Weight Decay** ($0, 10^{-4}, 10^{-2}$), **Loss Function** (MSE vs. Huber), and **Activation** (ReLU, GELU, SiLU) on validation RMSE.

In [4]:
df_ablation = pd.read_csv(DL_ROOT / "results" / "tables" / "ablation_results.csv")
display(df_ablation)

## 4. Final 5-Seed Evaluation, Uncertainty Quantification & Leakage-Aware Ablation (Tasks E & F)
We evaluate five final model variants across 5 random seeds (`[42, 43, 44, 45, 46]`):
- `dl_mlp`: Compact regularized MLP (`[128, 64, 32]`, ReLU, BatchNorm, Dropout 0.1, Huber loss, 13,441 params)
- `dl_mlp_mcdropout`: Monte Carlo Dropout with $T=50$ stochastic forward passes per sample (BatchNorm frozen in `.eval()`)
- `dl_mlp_ensemble`: 5-Seed Deep Ensemble combining epistemic inter-seed variance with split-conformal calibration
- `dl_mlp_quantile`: 3-Head Pinball Quantile Regression MLP ($q_{05}, q_{50}, q_{95}$) with non-crossing regularization
- `dl_mlp_leakaware`: Controlled ablation with `leakage_aware=true` (dropping the 4 direct stoichiometric proxies `do_deficit`, `log_np_ratio`, `nitrate_to_n_ratio`, `ph_dist_buffer`)

In [5]:
df_final = pd.read_csv(DL_ROOT / "results" / "tables" / "phase2_final_summary.csv")
display(df_final)

unc_img = DL_ROOT / "results" / "figures" / "mlp_uncertainty_calibration.png"
if unc_img.exists():
    display(Image(filename=str(unc_img)))